# <b>RideOps-Intelligence-Dashboard</b>

In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [59]:
data=pd.read_csv("ncr_ride_bookings.csv")
data.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


### <b>EDA and Data Visualisation</b>

In [60]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

#### Converting the <i>Date column</i> and <i>Time column</i> into data and time format

In [61]:
data['Date'] = pd.to_datetime(data['Date'])
data['Time'] = pd.to_datetime(data['Time']).dt.time

C:\Users\rajat\AppData\Local\Temp\ipykernel_17704\2138348502.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Time'] = pd.to_datetime(data['Time']).dt.time


##### Creating day_period out of the time column to get analysis according to differet time periods

In [62]:
def day_perd(time):
    hour=time.hour
    
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

data['Day_Period'] = data['Time'].apply(day_perd)

In [63]:
data.head(2)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Day_Period
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afternoon
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,Evening


##### Creating Season out of the Date column to get analysis according to differet seasons

In [64]:
def season(time):
    month=time.month
    
    if month in [11,12,1,2]:
        return 'Winter'
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return 'Autumn'
    
data['Season']=data['Date'].apply(season)

In [65]:
data.head(3)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Day_Period,Season
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afternoon,Spring
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,Evening,Winter
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,Morning,Summer


#### Handling and fixing null values

In [66]:
data['Booking Status'].unique()

array(['No Driver Found', 'Incomplete', 'Completed',
       'Cancelled by Driver', 'Cancelled by Customer'], dtype=object)

`Booking Status`=<b>Completed</b>

In [67]:
data.loc[data['Booking Status']=="Completed",
         ['Cancelled Rides by Customer','Reason for cancelling by Customer','Cancelled Rides by Driver',
          'Driver Cancellation Reason', 'Incomplete Rides','Incomplete Rides Reason']
         ]=data.loc[data['Booking Status']=="Completed",
                    ['Cancelled Rides by Customer','Reason for cancelling by Customer','Cancelled Rides by Driver',
                     'Driver Cancellation Reason', 'Incomplete Rides','Incomplete Rides Reason']].fillna(
           {
            'Cancelled Rides by Customer':0,
            'Reason for cancelling by Customer':"Not Cancelled",
            'Cancelled Rides by Driver':0,
            'Driver Cancellation Reason':"Not Cancelled", 
            'Incomplete Rides':0,
            'Incomplete Rides Reason':"Ride Completed"
           }
       )

In [68]:
data[data['Booking Status']=='Completed']

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Day_Period,Season
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,Not Cancelled,0.0,Ride Completed,627.0,13.58,4.9,4.9,Debit Card,Morning,Summer
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,Not Cancelled,0.0,Ride Completed,416.0,34.02,4.6,5.0,UPI,Evening,Autumn
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,Not Cancelled,0.0,Ride Completed,737.0,48.21,4.1,4.3,UPI,Night,Autumn
5,2024-02-06,09:44:56,"""CNR4096693""",Completed,"""CID4670564""",Auto,AIIMS,Narsinghpur,5.1,18.1,...,Not Cancelled,0.0,Ride Completed,316.0,4.85,4.1,4.6,UPI,Morning,Winter
6,2024-06-17,15:45:58,"""CNR2002539""",Completed,"""CID6800553""",Go Mini,Vaishali,Punjabi Bagh,7.1,20.4,...,Not Cancelled,0.0,Ride Completed,640.0,41.24,4.0,4.1,UPI,Afternoon,Summer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,2024-11-11,19:34:01,"""CNR6500631""",Completed,"""CID4337371""",Go Mini,MG Road,Ghitorni,10.2,44.4,...,Not Cancelled,0.0,Ride Completed,475.0,40.08,3.7,4.1,Uber Wallet,Evening,Winter
149996,2024-11-24,15:55:09,"""CNR2468611""",Completed,"""CID2325623""",Go Mini,Golf Course Road,Akshardham,5.1,30.8,...,Not Cancelled,0.0,Ride Completed,1093.0,21.31,4.8,5.0,UPI,Afternoon,Winter
149997,2024-09-18,10:55:15,"""CNR6358306""",Completed,"""CID9925486""",Go Sedan,Satguru Ram Singh Marg,Jor Bagh,2.7,23.4,...,Not Cancelled,0.0,Ride Completed,852.0,15.93,3.9,4.4,Cash,Morning,Autumn
149998,2024-10-05,07:53:34,"""CNR3030099""",Completed,"""CID9415487""",Auto,Ghaziabad,Saidulajab,6.9,39.6,...,Not Cancelled,0.0,Ride Completed,333.0,45.54,4.1,3.7,UPI,Morning,Autumn


`Booking Status`=<b>Cancelled by Customer</b>

In [72]:
data.loc[data['Booking Status']=="Cancelled by Customer",
         [
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating','Payment Method']
         ]=data.loc[data['Booking Status']=="Cancelled by Customer",
                    [
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating','Payment Method']].fillna(
           {
       'Avg CTAT':np.nan,
       'Cancelled Rides by Customer':1, 
       'Cancelled Rides by Driver':0,
       'Driver Cancellation Reason':"Not Cancelled", 
       'Incomplete Rides':0,
       'Incomplete Rides Reason':"Not Cancelled", 
       'Booking Value':np.nan,
       'Ride Distance':np.nan,
       'Driver Ratings':np.nan,
       'Customer Rating':np.nan,
       'Payment Method':"No Payment"
           }
       )

In [74]:
data[data['Booking Status']=='Cancelled by Customer']

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Day_Period,Season
18,2024-11-24,09:07:10,"""CNR6126048""",Cancelled by Customer,"""CID1060329""",eBike,Kashmere Gate,Anand Vihar,12.4,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Winter
39,2024-09-10,13:02:42,"""CNR4218487""",Cancelled by Customer,"""CID3037053""",Bike,Noida Extension,Udyog Vihar Phase 4,11.0,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Afternoon,Autumn
46,2024-08-02,07:17:07,"""CNR4862806""",Cancelled by Customer,"""CID7875150""",Auto,Shastri Park,Anand Vihar ISBT,11.3,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Summer
62,2024-02-09,11:15:59,"""CNR2497989""",Cancelled by Customer,"""CID5007066""",Auto,Karkarduma,IGI Airport,16.6,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Winter
78,2024-11-25,08:29:14,"""CNR2601752""",Cancelled by Customer,"""CID9283370""",Go Sedan,Sadar Bazar Gurgaon,Indraprastha,12.0,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149943,2024-05-01,16:39:56,"""CNR3486851""",Cancelled by Customer,"""CID7572575""",Bike,IGNOU Road,Kadarpur,7.8,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Afternoon,Spring
149952,2024-04-10,09:33:06,"""CNR8257559""",Cancelled by Customer,"""CID1017725""",Auto,Pitampura,IGNOU Road,6.4,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Spring
149956,2024-10-10,18:34:10,"""CNR6030764""",Cancelled by Customer,"""CID6873715""",Go Sedan,Hauz Rani,Dilshad Garden,13.2,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Evening,Autumn
149963,2024-02-04,10:51:54,"""CNR4272547""",Cancelled by Customer,"""CID2998831""",Auto,Jahangirpuri,Lal Quila,9.7,NaN,...,Not Cancelled,0.0,Not Cancelled,NaN,NaN,NaN,NaN,No Payment,Morning,Winter


In [ ]:
# data.to_csv("cleaed_rides.csv",index=False)